# Milvus VectorStore

## 학습 목표

이 노트북을 마치면 다음을 할 수 있어요:

- Milvus의 특징과 다른 VectorStore와의 차이를 설명할 수 있어요
- Milvus Lite로 로컬에서 빠르게 시작하고 Docker로 운영 환경에 배포할 수 있어요
- 메타데이터 필터링, MMR 검색, RAG 체인 통합을 구현할 수 있어요

## 사전 지식

- 노트북 01~03(Chroma, FAISS, Pinecone) 학습 완료
- OpenAI API 키 설정

---

## 데이터 규모별 VectorStore 선택 가이드

```mermaid
flowchart TD
    S["데이터 규모 결정"]:::input
    A{"문서 수"}:::process
    B["Chroma<br/>소규모 개발·테스트<br/>수천~수만 문서"]:::output
    C["FAISS<br/>중규모 단일 서버<br/>수만~수백만 문서"]:::output
    D["Milvus<br/>중대규모 온프레미스<br/>수백만~수십억 문서"]:::output
    E["Pinecone<br/>초대규모 클라우드<br/>무제한 (관리형)"]:::external

    S --> A
    A -->|"< 10만"| B
    A -->|"10만 ~ 100만"| C
    A -->|"100만 ~ 수십억"| D
    A -->|"비용보다 편의"| E

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
    classDef external fill:#d1ecf1,stroke:#17a2b8,color:#0c5460
```

## Milvus의 특징

- 수십억 개 벡터를 밀리초 단위로 검색
- HNSW, IVF, DiskANN 등 다양한 인덱스 알고리즘
- 분산 아키텍처로 클러스터 구성 가능
- GPU 가속 지원
- 오픈소스 (무료)

**참고 문서**: [Milvus 공식 문서](https://milvus.io/docs) | [LangChain Milvus 통합](https://python.langchain.com/docs/integrations/vectorstores/milvus/)

## 설치 및 실행

### Milvus Lite (학습용, 권장)

Docker 없이 Python 패키지만으로 실행하는 경량 버전이에요. 이 노트북에서 사용해요.

```bash
pip install pymilvus
```

### Milvus Standalone (운영용)

Docker로 실행하는 완전한 Milvus 서버예요.

```bash
# docker-compose 파일 다운로드 및 실행
wget https://github.com/milvus-io/milvus/releases/download/v2.3.0/milvus-standalone-docker-compose.yml -O docker-compose.yml
docker-compose up -d
```

기본 포트: `http://localhost:19530`

## 환경 설정

In [ ]:
from dotenv import load_dotenv
import os

# API 키 정보 로드
load_dotenv()

print("✅ 환경 설정 완료")

---

## 1. 데이터 준비

In [ ]:

# ---------------------------------------------------
# 문서 로드 및 분할 (Milvus에 저장할 데이터 준비)
# ---------------------------------------------------

from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 텍스트 분할 설정
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)

# 1단계: 문서 로더 생성
loader_nlp = TextLoader("data/nlp-keywords.txt")
loader_finance = TextLoader("data/finance-keywords.txt")

# 2단계: 문서 로드 및 분할
docs_nlp = loader_nlp.load_and_split(text_splitter)
docs_finance = loader_finance.load_and_split(text_splitter)

# 3단계: 문서 개수 확인
print(f"NLP 문서: {len(docs_nlp)}개")
print(f"금융 문서: {len(docs_finance)}개")
print(f"\n[NLP 문서 샘플]\n{docs_nlp[0].page_content[:200]}...")


---

## 2. Milvus VectorStore 생성

### 2.1 from_documents로 생성

`connection_args={"uri": "./milvus_demo.db"}`를 지정하면 Milvus Lite(파일 기반)로 실행돼요. Docker 없이 로컬 파일로 저장돼요.

> **실무 팁**: 개발 단계에서는 파일 기반 URI, 운영 환경에서는 Docker 서버 주소(`"host": "localhost", "port": "19530"`)로 바꾸면 돼요. 코드 변경은 최소화돼요.

In [ ]:
# ---------------------------------------------------
# Milvus VectorStore 생성 (Milvus Lite 파일 기반)
# ---------------------------------------------------

# ============================================================
# TODO: Milvus.from_documents()로 벡터 저장소를 생성해보세요
# 힌트: Milvus.from_documents(docs, embedding, collection_name=..., connection_args={"uri": "./milvus_demo.db"})
# 예상 결과: "NLP VectorStore 생성 완료" 출력
# ============================================================

from langchain_community.vectorstores import Milvus
from langchain_openai import OpenAIEmbeddings

# 1단계: 임베딩 모델 초기화
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# 2단계: Milvus VectorStore 생성 (Milvus Lite — 파일 기반)
# connection_args={"uri": "./milvus_demo.db"}: 로컬 파일에 저장 (Docker 불필요)
# collection_name: 컬렉션 이름 (Milvus의 테이블 개념)


print("✅ NLP VectorStore 생성 완료")
print(f"- 컬렉션: nlp_collection")
print(f"- 문서 수: {len(docs_nlp)}개")

### 2.2 기존 컬렉션 연결

이미 만들어진 컬렉션에 재연결할 수 있어요.

In [ ]:

# ---------------------------------------------------
# 기존 Milvus 컬렉션에 재연결
# ---------------------------------------------------

# Milvus(): from_documents() 없이 이미 존재하는 컬렉션에 연결
# 임베딩 재계산 없이 기존 데이터를 바로 사용할 수 있음
vectorstore_existing = Milvus(
    embedding_function=embeddings,
    collection_name="nlp_collection",
    connection_args={"uri": "./milvus_demo.db"}
)

print("✅ 기존 컬렉션 연결 완료")


---

## 3. 유사도 검색

### 3.1 기본 유사도 검색

In [ ]:
# ---------------------------------------------------
# 기본 유사도 검색
# ---------------------------------------------------

# ============================================================
# TODO: similarity_search()로 쿼리와 유사한 문서를 검색해보세요
# 힌트: vectorstore_nlp.similarity_search(query, k=3)
# 예상 결과: 자연어 처리/딥러닝 관련 문서 3개 반환
# ============================================================

query = "자연어 처리와 딥러닝"

# similarity_search(): Milvus도 Chroma/FAISS와 동일한 API 사용


print(f"검색어: '{query}'")
print("=" * 60)
for i, doc in enumerate(results, 1):
    print(f"\n[결과 {i}]")
    print(doc.page_content[:200])
    print(f"메타데이터: {doc.metadata}")

### 3.2 점수 포함 검색

`similarity_search_with_score()`는 유사도 점수와 함께 결과를 반환해요. Milvus는 기본적으로 거리(distance)를 반환해요 — 값이 낮을수록 유사해요.

In [ ]:

# ---------------------------------------------------
# 유사도 점수 포함 검색
# ---------------------------------------------------

# similarity_search_with_score(): 문서와 함께 거리(distance) 점수 반환
# Milvus는 거리 기반 — 값이 낮을수록 더 유사한 문서
results_with_scores = vectorstore_nlp.similarity_search_with_score(query, k=3)

print(f"검색어: '{query}'")
print("=" * 60)
for i, (doc, score) in enumerate(results_with_scores, 1):
    print(f"\n[결과 {i}] 유사도 거리: {score:.4f}")  # 낮을수록 유사
    print(doc.page_content[:150])


### 3.3 MMR 검색 (다양성 고려)

MMR(Maximum Marginal Relevance) 검색으로 유사도와 다양성을 균형 있게 고려해볼게요.

In [ ]:
# ---------------------------------------------------
# MMR 검색 (다양성 고려)
# ---------------------------------------------------

# ============================================================
# TODO: max_marginal_relevance_search()로 MMR 검색을 수행해보세요
# 힌트: vectorstore_nlp.max_marginal_relevance_search(query, k=4, fetch_k=20)
# 예상 결과: 유사도만 고려했을 때보다 다양한 주제의 문서 4개 반환
# ============================================================

# max_marginal_relevance_search(): 유사도와 다양성을 동시에 고려
# fetch_k: 먼저 유사도 기준 후보 선정 → 이 중 k개를 다양성 기준으로 최종 선택


print(f"MMR 검색어: '{query}'")
print("=" * 60)
for i, doc in enumerate(mmr_results, 1):
    print(f"\n[MMR 결과 {i}]")
    print(doc.page_content[:150])

---

## 4. 메타데이터 필터링

Milvus는 SQL과 유사한 `expr` 표현식으로 메타데이터 필터링을 지원해요. Chroma의 `filter`보다 더 강력한 조건식을 작성할 수 있어요.

In [ ]:

# ---------------------------------------------------
# 메타데이터 포함 문서 추가 (필터링 테스트용)
# ---------------------------------------------------

from langchain_core.documents import Document

# 메타데이터 필드를 포함한 문서 생성
docs_with_metadata = [
    Document(
        page_content="Transformer는 2017년에 발표된 딥러닝 모델입니다.",
        metadata={"year": 2017, "category": "architecture", "importance": "high"}
    ),
    Document(
        page_content="BERT는 2018년 Google이 발표한 사전학습 모델입니다.",
        metadata={"year": 2018, "category": "pretrained", "importance": "high"}
    ),
    Document(
        page_content="Word2Vec은 2013년 단어 임베딩 기법입니다.",
        metadata={"year": 2013, "category": "embedding", "importance": "medium"}
    ),
]

# 1단계: 메타데이터 포함 문서를 새 컬렉션에 저장
vectorstore_filtered = Milvus.from_documents(
    documents=docs_with_metadata,
    embedding=embeddings,
    collection_name="nlp_with_metadata",
    connection_args={"uri": "./milvus_demo.db"}
)

print("✅ 메타데이터 포함 문서 추가 완료")

In [ ]:

# ---------------------------------------------------
# 메타데이터 필터링 검색 안내
# ---------------------------------------------------

# Milvus는 SQL과 유사한 expr 표현식으로 필터 조건을 지정해요
# 예: expr="year >= 2017 and importance == 'high'"
# 이 기능은 Milvus가 Chroma의 딕셔너리 필터보다 더 강력한 조건식을 지원한다는 점이 특징이에요
print("💡 Milvus 메타데이터 필터링 예시:")
print("  expr=\"year >= 2017\"")
print("  expr=\"category == 'pretrained'\"")
print("  expr=\"year >= 2017 and importance == 'high'\"")


---

## 5. 문서 추가 및 삭제

### 5.1 문서 추가

In [ ]:

# ---------------------------------------------------
# 기존 컬렉션에 새 문서 추가
# ---------------------------------------------------

new_docs = [
    Document(
        page_content="GPT-4는 2023년에 발표된 대규모 언어 모델입니다.",
        metadata={"year": 2023, "category": "llm"}
    )
]

# add_documents(): 기존 컬렉션에 새 문서 추가 (전체 재생성 불필요)
ids = vectorstore_filtered.add_documents(new_docs)

print(f"✅ 문서 추가 완료")
print(f"추가된 문서 ID: {ids}")


In [ ]:
# 추가 확인
results = vectorstore_filtered.similarity_search("대규모 언어 모델", k=2)
print("검색 결과:")
for doc in results:
    print(f"- {doc.page_content}")

### 5.2 문서 삭제

In [ ]:
# ID로 문서 삭제
if ids:
    vectorstore_filtered.delete(ids)
    print(f"✅ 문서 삭제 완료: {ids}")

---

## 6. Retriever로 변환

RAG 체인에 사용하기 위해 Retriever로 변환해볼게요.

In [ ]:
# ---------------------------------------------------
# Milvus VectorStore를 Retriever로 변환
# ---------------------------------------------------

# ============================================================
# TODO: as_retriever()로 Retriever를 만들고 검색을 실행해보세요
# 힌트: vectorstore_nlp.as_retriever(search_type="similarity", search_kwargs={"k": 4})
# 예상 결과: "자연어 처리" 관련 문서 4개 반환
# ============================================================

# 1단계: Retriever 생성
# search_type="similarity": 코사인 유사도 기반 검색 (기본값)


# 2단계: 검색 실행
retrieved_docs = retriever.invoke("자연어 처리")

print(f"✅ Retriever 생성 완료")
print(f"검색된 문서 수: {len(retrieved_docs)}개")
print(f"\n첫 번째 문서:\n{retrieved_docs[0].page_content[:150]}...")

---

## 7. RAG 체인 구축

Milvus Retriever를 활용한 완전한 RAG 시스템을 구성해볼게요.

In [ ]:

# ---------------------------------------------------
# Milvus Retriever를 활용한 RAG 체인 구성
# ---------------------------------------------------

from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# 1단계: 프롬프트 템플릿 정의
prompt = PromptTemplate.from_template(
    """문맥을 참고하여 질문에 답하세요.

문맥:
{context}

질문: {question}

답변:"""
)

# 2단계: LLM 초기화
# temperature=0: 동일 입력에 항상 동일 출력 (재현성 확보)
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# 3단계: RAG 체인 구성 (파이프(|) 연산자: 왼쪽 출력이 오른쪽 입력으로 전달)
rag_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("✅ RAG 체인 구축 완료")


In [ ]:

# ---------------------------------------------------
# RAG 체인 실행 — 질문에 대한 답변 생성
# ---------------------------------------------------

question = "트랜스포머 아키텍처에 대해 설명해주세요."

print(f"질문: {question}")
print("=" * 60)
# invoke(): 체인의 끝에서 최종 답변(문자열)을 반환
answer = rag_chain.invoke(question)
print(f"답변:\n{answer}")


---

## 8. Docker Milvus 연결 (운영 환경)

Docker로 Milvus Standalone을 실행하는 경우 연결 방법을 확인해볼게요. Milvus Lite와 코드가 거의 동일해요.

In [ ]:
# Docker Milvus 연결 예시 (실제 실행하지 않음)
'''
# Docker로 Milvus 실행 시
vectorstore_docker = Milvus.from_documents(
    documents=docs_nlp,
    embedding=embeddings,
    collection_name="nlp_docker",
    connection_args={
        "host": "localhost",  # Milvus 서버 주소
        "port": "19530"  # 기본 포트
    }
)
'''

print("💡 Docker Milvus 연결 방법을 주석으로 확인하세요.")

---

## 9. 컬렉션 정보 확인

`pymilvus`로 저장된 벡터 수와 스키마를 직접 확인할 수 있어요.

In [ ]:

# ---------------------------------------------------
# pymilvus로 컬렉션 통계 직접 조회
# ---------------------------------------------------

from pymilvus import connections, Collection

# 1단계: Milvus Lite 파일에 연결
connections.connect(uri="./milvus_demo.db")

# 2단계: 컬렉션 객체 가져오기
collection = Collection("nlp_collection")

# 3단계: 컬렉션 통계 출력
# num_entities: 저장된 벡터(문서) 수
print(f"컬렉션명: {collection.name}")
print(f"저장된 벡터 수: {collection.num_entities}개")
print(f"스키마: {collection.schema}")


---

## 마무리

### VectorStore 최종 비교

| 특징 | Chroma | FAISS | Milvus | Pinecone |
|------|--------|-------|--------|----------|
| 적합한 규모 | 소규모 | 중규모 | 대규모 | 무제한 |
| 설치 난이도 | 쉬움 | 쉬움 | 보통 | 가입만 |
| 비용 | 무료 | 무료 | 무료 | 유료 |
| 분산 지원 | 없음 | 없음 | 있음 | 있음 |
| GPU 가속 | 없음 | 제한적 | 있음 | 있음 |
| 추천 환경 | 개발 | 단일 서버 | 온프레미스 대규모 | 클라우드 운영 |

### 선택 기준 요약

- 10만 문서 이하: **Chroma** (설치가 가장 쉬워요)
- 10만~100만: **FAISS** (빠르고 무료예요)
- 100만~수십억, 자체 서버: **Milvus** (오픈소스 고성능)
- 인프라 관리 없이 대규모: **Pinecone** (완전 관리형)

### 다음 학습

**6-5 Retriever**: VectorStore를 다양한 검색 전략과 결합하는 고급 Retriever 기법을 배워볼게요.

## 정리 (Cleanup)

In [ ]:
# 연결 종료
connections.disconnect("default")
print("✅ Milvus 연결 종료")